# 🚨 Churn Prediction Model

> **Dataset:** Online Retail (UCI Machine Learning Repository)  
> **Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
> **Goal:** Build a machine learning model that predicts which customers are likely to churn, identify the most important churn drivers, and produce an actionable priority list for retention teams.

---

## Business Question

> *"Which of our active customers are most likely to stop buying — and what signals tell us before it happens?"*

### What is Churn Here?
In a non-contractual e-commerce setting, churn is not observable directly. We define a customer as **churned** if they have made **no purchase in the last 90 days** of the observation window — a common industry threshold for this type of retailer.

### Modelling Approach

| Step | Method |
|---|---|
| Feature engineering | RFM metrics + behavioural signals |
| Baseline | Logistic Regression |
| Main model | Random Forest Classifier |
| Evaluation | ROC-AUC, Precision, Recall, Confusion Matrix |
| Output | Churn probability score per customer + feature importance |

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay
)

# ── Palette ─────────────────────────────────────────────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'
GREEN_OK   = '#5a8a5e'

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)
print('Libraries loaded ✅')

---
## 2. Load & Clean Data

In [ ]:
df = pd.read_excel('data/online_retail.xlsx', dtype={'CustomerID': str})

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df = df[df['Country'] == 'United Kingdom'].copy()

print(f'Clean rows:      {len(df):,}')
print(f'Unique customers:{df.CustomerID.nunique():,}')

---
## 3. Define Churn & Feature Engineering

We split the observation window into:
- **Feature window:** all history up to 90 days before the end
- **Outcome window:** the final 90 days — did the customer purchase?

This prevents data leakage and mirrors a real deployment scenario.

In [ ]:
snapshot = df['InvoiceDate'].max()
cutoff   = snapshot - pd.Timedelta(days=90)

# ── Feature window ────────────────────────────────────────────
df_feat = df[df['InvoiceDate'] < cutoff].copy()

# ── Outcome window ────────────────────────────────────────────
active_in_outcome = set(
    df[df['InvoiceDate'] >= cutoff]['CustomerID'].unique()
)

# ── RFM features ─────────────────────────────────────────────
features = df_feat.groupby('CustomerID').agg(
    Recency         = ('InvoiceDate', lambda x: (cutoff - x.max()).days),
    Frequency       = ('InvoiceNo', 'nunique'),
    MonetaryValue   = ('Revenue', 'sum'),
    AvgOrderValue   = ('Revenue', lambda x: x.groupby(df_feat.loc[x.index,'InvoiceNo']).sum().mean()),
    NumItems        = ('Quantity', 'sum'),
    UniqueProducts  = ('StockCode', 'nunique'),
    TenureDays      = ('InvoiceDate', lambda x: (x.max() - x.min()).days),
    AvgDaysBetween  = ('InvoiceDate', lambda x: x.sort_values().diff().dt.days.mean()),
).reset_index()

# ── Target: 1 = churned (no purchase in outcome window) ──────
features['Churned'] = (~features['CustomerID'].isin(active_in_outcome)).astype(int)

# ── Fill NaN avg days between (single-purchase customers) ────
features['AvgDaysBetween'] = features['AvgDaysBetween'].fillna(features['TenureDays'])

print(f'Customers in feature set: {len(features):,}')
print(f'Churned:     {features.Churned.sum():,} ({features.Churned.mean():.1%})')
print(f'Retained:    {(1-features.Churned).sum():,} ({(1-features.Churned).mean():.1%})')
features.head()

---
## 4. Exploratory Analysis — Churned vs Retained

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

feature_cols = ['Recency','Frequency','MonetaryValue','AvgOrderValue','UniqueProducts','AvgDaysBetween']
labels = ['Recency (days)','Frequency (orders)','Total Spend (£)','Avg Order Value (£)','Unique Products','Avg Days Between Orders']

for ax, col, label in zip(axes, feature_cols, labels):
    churned  = features[features['Churned']==1][col].clip(upper=features[col].quantile(0.95))
    retained = features[features['Churned']==0][col].clip(upper=features[col].quantile(0.95))
    ax.hist(retained, bins=35, alpha=0.65, color=GREEN_OK,   label='Retained', density=True)
    ax.hist(churned,  bins=35, alpha=0.65, color=TERRACOTTA, label='Churned',  density=True)
    ax.set_title(label)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.grid(axis='y')

plt.suptitle('Feature Distributions: Churned vs Retained', fontsize=14, color=CHOCOLATE, y=1.01)
plt.tight_layout()
plt.savefig('outputs/churn_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Model Training

In [ ]:
FEATURE_COLS = ['Recency','Frequency','MonetaryValue','AvgOrderValue',
                'NumItems','UniqueProducts','TenureDays','AvgDaysBetween']

X = features[FEATURE_COLS]
y = features['Churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Logistic Regression (baseline) ───────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1])

# ── Random Forest ─────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

# ── Gradient Boosting ─────────────────────────────────────────
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                 max_depth=4, random_state=42)
gb.fit(X_train, y_train)
gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:,1])

print(f'Logistic Regression ROC-AUC: {lr_auc:.4f}')
print(f'Random Forest       ROC-AUC: {rf_auc:.4f}')
print(f'Gradient Boosting   ROC-AUC: {gb_auc:.4f}')

# Use best model
best_model = max([(lr_auc, lr, 'Logistic Regression', X_test_sc),
                  (rf_auc, rf, 'Random Forest', X_test),
                  (gb_auc, gb, 'Gradient Boosting', X_test)],
                 key=lambda x: x[0])
best_auc, best_clf, best_name, best_X = best_model
print(f'\n✅ Best model: {best_name} (AUC: {best_auc:.4f})')

---
## 6. Model Evaluation

In [ ]:
y_pred      = best_clf.predict(best_X)
y_pred_prob = best_clf.predict_proba(best_X)[:,1]

print(f'=== {best_name} — Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Confusion matrix ──────────────────────────────────────────
ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned'])
disp.plot(ax=ax, cmap='YlOrBr', colorbar=False)
ax.set_title(f'Confusion Matrix\n{best_name}')

# ── ROC curve ─────────────────────────────────────────────────
ax2 = axes[1]
for auc_score, model, name, Xeval in [
    (lr_auc, lr, 'Logistic Regression', X_test_sc),
    (rf_auc, rf, 'Random Forest', X_test),
    (gb_auc, gb, 'Gradient Boosting', X_test)
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(Xeval)[:,1])
    color = CHOCOLATE if name == best_name else SAND
    lw    = 2.5 if name == best_name else 1
    ax2.plot(fpr, tpr, color=color, lw=lw, label=f'{name} (AUC={auc_score:.3f})')
ax2.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.4)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve — All Models')
ax2.legend(fontsize=8, framealpha=0.7)
ax2.grid(True, alpha=0.4)

# ── Precision-Recall curve ────────────────────────────────────
ax3 = axes[2]
prec, rec, _ = precision_recall_curve(y_test, y_pred_prob)
ax3.plot(rec, prec, color=CHOCOLATE, lw=2)
ax3.fill_between(rec, prec, alpha=0.15, color=CAMEL)
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title(f'Precision-Recall Curve\n{best_name}')
ax3.grid(True, alpha=0.4)

plt.suptitle('Model Evaluation', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/churn_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Feature Importance

In [ ]:
# Use Random Forest for feature importance (tree-based)
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [CHOCOLATE if v == importances.max() else
          BROWN if v >= importances.quantile(0.75) else
          CAMEL if v >= importances.quantile(0.5) else SAND
          for v in importances.values]

bars = ax.barh(importances.index, importances.values,
               color=colors, edgecolor='white', linewidth=0.4, height=0.6)

for bar, v in zip(bars, importances.values):
    ax.text(v + 0.002, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=9, color=CHOCOLATE)

ax.set_title('Feature Importance — Random Forest Churn Model')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.savefig('outputs/churn_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 churn drivers:')
for feat, val in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {val:.3f}')

---
## 8. Churn Probability Scores & Priority List

In [ ]:
# Score ALL customers (not just test set)
features['ChurnProbability'] = rf.predict_proba(features[FEATURE_COLS])[:,1]

# Risk bands
def risk_band(p):
    if p >= 0.75:   return 'Critical'
    elif p >= 0.50: return 'High'
    elif p >= 0.25: return 'Medium'
    else:           return 'Low'

features['RiskBand'] = features['ChurnProbability'].apply(risk_band)

band_order = ['Critical','High','Medium','Low']
band_summary = features.groupby('RiskBand').agg(
    Customers        = ('CustomerID', 'count'),
    AvgChurnProb     = ('ChurnProbability', 'mean'),
    AvgSpend         = ('MonetaryValue', 'mean'),
    TotalRevenueAtRisk = ('MonetaryValue', 'sum')
).reindex(band_order).round(2)

band_summary['TotalRevenueAtRisk'] = band_summary['TotalRevenueAtRisk'].map('£{:,.0f}'.format)
band_summary['AvgSpend'] = band_summary['AvgSpend'].map('£{:,.0f}'.format)
band_summary['AvgChurnProb'] = band_summary['AvgChurnProb'].map('{:.1%}'.format)
print('Risk Band Summary:')
band_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

band_colors = {'Critical': TERRACOTTA, 'High': BROWN, 'Medium': CAMEL, 'Low': SAND}

# Churn probability distribution
ax = axes[0]
ax.hist(features['ChurnProbability'], bins=40,
        color=CHOCOLATE, edgecolor='white', linewidth=0.3, alpha=0.85)
for thresh, label, color in zip(
    [0.25, 0.50, 0.75],
    ['Medium', 'High', 'Critical'],
    [CAMEL, BROWN, TERRACOTTA]
):
    ax.axvline(thresh, color=color, linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(thresh+0.01, ax.get_ylim()[1]*0.85, label,
            fontsize=8, color=color, style='italic')
ax.set_title('Churn Probability Distribution')
ax.set_xlabel('Predicted Churn Probability')
ax.set_ylabel('Customers')
ax.grid(axis='y')

# Risk band customer count
ax2 = axes[1]
band_counts = features['RiskBand'].value_counts().reindex(band_order)
bars = ax2.bar(band_order, band_counts.values,
               color=[band_colors[b] for b in band_order],
               edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, band_counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
             f'{v:,}', ha='center', fontsize=9, color=CHOCOLATE)
ax2.set_title('Customers by Churn Risk Band')
ax2.set_ylabel('Customers')
ax2.grid(axis='y')

plt.suptitle('Churn Risk Segmentation', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/churn_risk_bands.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Export

In [ ]:
output = features[[
    'CustomerID','Recency','Frequency','MonetaryValue','AvgOrderValue',
    'UniqueProducts','TenureDays','ChurnProbability','RiskBand','Churned'
]].sort_values('ChurnProbability', ascending=False)

output.to_csv('outputs/churn_scores.csv', index=False)
print(f'Exported {len(output):,} customer churn scores → outputs/churn_scores.csv ✅')
print(f'\nTop 10 highest-risk customers:')
output.head(10)[['CustomerID','MonetaryValue','ChurnProbability','RiskBand']]

---
## 10. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **Recency is the strongest churn predictor** — customers who haven't bought recently are far more likely to churn, regardless of historical spend |
| 2 | **Frequency and Tenure protect against churn** — long-standing, frequent buyers show significantly lower churn probability |
| 3 | **High-value customers are not immune** — a meaningful share of Critical-risk customers have high historical spend, making them disproportionately valuable to save |
| 4 | **The model outperforms random targeting** — ROC-AUC well above 0.5 confirms the model is learning real signal |
| 5 | **Average Order Value is a weak churn signal** — how often and how recently customers buy matters more than how much per transaction |

---

### 💡 Recommendations

**1. Prioritise Critical-band customers with high historical spend**  
Sort the exported `churn_scores.csv` by `ChurnProbability DESC` and `MonetaryValue DESC`. These customers represent the highest revenue at risk and should receive immediate, personalised outreach.

**2. Automate recency-triggered re-engagement**  
Since Recency is the top driver, set up an automated email triggered at 30, 60, and 90 days post-last-purchase. The 30-day touch is most cost-effective — catching customers before they reach Critical risk.

**3. Do not treat all churn risk equally**  
Low-band customers with low spend are not worth expensive retention spend. Focus campaign budget on Critical and High bands — especially those with Frequency ≥ 3 (proven buyers worth recovering).

**4. Retrain the model quarterly**  
Purchasing behaviour shifts seasonally. Retrain every 3 months using a rolling observation window to keep churn probabilities accurate.

**5. Connect churn scores to the LTV model**  
Multiply `ChurnProbability × PredictedLTV` to produce a **revenue-at-risk score** per customer. This gives the retention team a single number for prioritisation that accounts for both likelihood and value.

---

*Analysis by Danai Avratoglou | Dataset: UCI Online Retail | Tools: Python, scikit-learn, pandas, matplotlib*